# Single Pendulum: Tabular Q-Learning

A complete value-based baseline. Run top-to-bottom, then tune only the configuration cell.
The table uses four compact coordinates: cart position, cart velocity, wrapped pole angle,
and pole angular velocity. Checkpoints and plots survive Colab disconnects when their root
points into Google Drive.


## 1. Runtime setup
No project repository is cloned. This notebook contains its own plant and algorithm.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive")

IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_COLAB or IN_KAGGLE:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--upgrade", "-q",
        "mujoco>=3.2", "gymnasium>=1.0", "pandas>=2.0", "matplotlib>=3.8",
        "imageio>=2.34", "imageio-ffmpeg>=0.5", "scipy>=1.11",
    ], check=True)

# Native MuJoCo dynamics are CPU-based; EGL uses the hosted NVIDIA GPU for video rendering.
os.environ.setdefault("MUJOCO_GL", "egl")
print("Runtime:", "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local")


## 2. Tuning and persistent paths
Change `OUTPUT_DIR` and hyperparameters here.


In [ ]:
OUTPUT_DIR = Path("/content/drive/MyDrive/ProjectsRuns/TIPy/runs/single/qlearning/run-001") if IN_COLAB else Path.cwd() / "qlearning-run"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
METRICS_PATH = OUTPUT_DIR / "metrics.csv"
DASHBOARD_PATH = OUTPUT_DIR / "dashboard.png"
TITLE = "TIPy Single Pendulum Q-Learning"
SEED = 42
EPISODES = 500
MAX_EPISODE_STEPS = 2000
ACTION_LIMIT = 100.0
ALPHA = 0.15
GAMMA = 0.99
EPSILON_START, EPSILON_END = 1.0, 0.05
EPSILON_DECAY_EPISODES = 400
BINS = (9, 9, 16, 12)
CHECKPOINT_EVERY = 25
SMOKE_TEST = False
if SMOKE_TEST:
    EPISODES, MAX_EPISODE_STEPS, CHECKPOINT_EVERY = 3, 100, 1
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Plant and environment
The terminal mask records rail failure; a time-limit truncation still bootstraps.


In [ ]:
import gymnasium as gym
from gymnasium import spaces
import mujoco
import numpy as np

MODEL_XML = r"""
<mujoco model="cartpole_single">
  <compiler angle="radian" autolimits="true" inertiafromgeom="false" />
  <option timestep="0.005" gravity="0 0 -9.81" integrator="RK4" />
  <default>
    <geom contype="0" conaffinity="0" />
  </default>
  <visual>
    <headlight diffuse="0.7 0.7 0.7" ambient="0.3 0.3 0.3" specular="0.1 0.1 0.1" />
    <rgba haze="0.15 0.2 0.25 1" />
  </visual>
  <asset>
    <material name="floor_mat" rgba="0.16 0.22 0.28 1" />
    <material name="metal_mat" rgba="0.65 0.68 0.72 1" />
    <material name="cart_mat" rgba="0.82 0.18 0.18 1" />
    <material name="pole_mat" rgba="0.2 0.75 0.3 1" />
    <material name="site_mat" rgba="0.95 0.9 0.15 0.9" />
    <material name="rail_limit_mat" rgba="0.95 0.55 0.05 1" />
  </asset>
  <worldbody>
    <camera name="spectate" pos="0 3 1.4" fovy="90" xyaxes="-1 0 0 0 -0.1240 0.9923" />
    <light diffuse="0.7 0.7 0.7" pos="0 0 3.5" dir="0 0 -1" />
    <geom name="floor" type="plane" size="5 5 0.1" material="floor_mat" />
    <body name="frame">
      <geom name="rail" type="box" pos="0 0 0.85" size="2.2 0.05 0.05" material="metal_mat" />
      <geom name="rail_limit_left" type="box" pos="-2.2 0 0.95" size="0.025 0.12 0.1" material="rail_limit_mat" />
      <geom name="rail_limit_right" type="box" pos="2.2 0 0.95" size="0.025 0.12 0.1" material="rail_limit_mat" />
    </body>
    <body name="cart" pos="0 0 1">
      <joint name="cart_slide" type="slide" axis="1 0 0" range="-2.2 2.2" frictionloss="0.02" damping="0.1" />
      <inertial pos="0 0 0" mass="2" diaginertia="0.0333 0.0333 0.0333" />
      <geom name="cart_geom" type="box" size="0.125 0.08 0.1" mass="2.0" material="cart_mat" />
      <site name="cart_center_site" pos="0 0 0" size="0.02" type="sphere" material="site_mat" />
      <geom name="mount_pin" type="capsule" pos="0 0.095 0" axisangle="1 0 0 1.5708" size="0.015 0.03" material="metal_mat" />
      <body name="pole" pos="0 0.11 0" quat="6.12323399574e-17 0 -1 0">
        <joint name="pole_hinge" type="hinge" axis="0 -1 0" frictionloss="0.01" damping="0.03" ref="3.1415926535897931" limited="false" />
        <inertial pos="0 0 0.3" mass="0.7" diaginertia="0.0210233333333 0.0210933333333 0.000116666666667" />
        <site name="pole_hinge_site" pos="0 0 0" size="0.015" type="sphere" material="site_mat" />
        <geom name="pole_geom" type="box" pos="0 0 0.3" size="0.02 0.01 0.3" mass="0.7" material="pole_mat" />
        <site name="pole_tip_site" pos="0 0 0.6" size="0.015" type="sphere" material="site_mat" />
      </body>
    </body>
    <camera name="replay" pos="0 6 1.4" fovy="50" xyaxes="-1 0 0 0 -0.15 0.988686" />
  </worldbody>
  <sensor>
    <jointpos name="cart_position" joint="cart_slide" />
    <jointpos name="pole1_relative_angle" joint="pole_hinge" />
    <jointvel name="cart_velocity" joint="cart_slide" />
    <jointvel name="pole1_relative_velocity" joint="pole_hinge" />
  </sensor>
  <actuator>
    <motor name="cart_motor" joint="cart_slide" gear="1" ctrlrange="-100 100" forcerange="-100 100" />
  </actuator>
</mujoco>
"""

def normalize_angle(theta):
    return float((theta + np.pi) % (2.0 * np.pi) - np.pi)

class SinglePendulumEnv(gym.Env):
    """Self-contained MuJoCo cart-pole swing-up environment."""
    def __init__(self, discrete=True, max_episode_steps=2000, action_limit=100.0):
        super().__init__()
        self.model = mujoco.MjModel.from_xml_string(MODEL_XML)
        self.data = mujoco.MjData(self.model)
        self.discrete = discrete
        self.max_episode_steps = max_episode_steps
        self.action_limit = float(action_limit)
        self.rail_limit = 2.2
        self.dt = float(self.model.opt.timestep)
        self.current_step = 0
        self.observation_space = spaces.Box(-1.0, 1.0, shape=(5,), dtype=np.float32)
        if discrete:
            self.action_table = np.linspace(-self.action_limit, self.action_limit, 7)
            self.action_space = spaces.Discrete(7)
        else:
            self.action_space = spaces.Box(-1.0, 1.0, shape=(1,), dtype=np.float32)

    def _observation(self):
        x, theta = map(float, self.data.qpos)
        dx, dtheta = map(float, self.data.qvel)
        theta = normalize_angle(theta)
        return np.array([
            np.clip(x / self.rail_limit, -1.0, 1.0),
            np.clip(dx / 5.0, -1.0, 1.0),
            np.cos(theta), np.sin(theta),
            np.clip(dtheta / 15.0, -1.0, 1.0),
        ], dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[:] = [self.np_random.uniform(-0.02, 0.02),
                             np.pi + self.np_random.uniform(-0.05, 0.05)]
        self.data.qvel[:] = self.np_random.uniform([-0.01, -0.02], [0.01, 0.02])
        mujoco.mj_forward(self.model, self.data)
        return self._observation(), {}

    def step(self, action):
        self.current_step += 1
        if self.discrete:
            force = float(self.action_table[int(action)])
        else:
            force = float(np.asarray(action).reshape(-1)[0]) * self.action_limit
        force = float(np.clip(force, -self.action_limit, self.action_limit))
        self.data.ctrl[0] = force
        mujoco.mj_step(self.model, self.data)
        x, theta = map(float, self.data.qpos)
        dx, dtheta = map(float, self.data.qvel)
        theta = normalize_angle(theta)
        reward = (np.cos(theta) - 0.2 * (x / self.rail_limit) ** 2
                  - 0.01 * dx ** 2 - 0.005 * dtheta ** 2
                  - 0.001 * (force / self.action_limit) ** 2
                  + (2.0 if abs(theta) < 0.35 else 0.0))
        terminated = bool(abs(x) >= self.rail_limit)
        if terminated:
            reward -= 10.0
        truncated = bool(self.current_step >= self.max_episode_steps)
        info = {"x": x, "theta": theta, "dx": dx, "dtheta": dtheta, "force": force}
        return self._observation(), float(reward), terminated, truncated, info

# Contract and reproducibility checks.
_env = SinglePendulumEnv(max_episode_steps=10)
_a, _ = _env.reset(seed=7)
_b, _ = _env.reset(seed=7)
assert _a.shape == (5,) and np.allclose(_a, _b)
assert _env.action_space.n == 7 and np.all(np.isfinite(_a))
print("Environment check passed; initial observation:", _a)


## 4. Q-learning agent
Tune bin counts before long runs; changing table shape starts an incompatible run.


In [ ]:
class QLearningAgent:
    def __init__(self, bins, actions, seed):
        self.bins, self.actions = tuple(bins), int(actions)
        self.q = np.zeros((*self.bins, self.actions), dtype=np.float32)
        self.epsilon = EPSILON_START
        self.rng = np.random.default_rng(seed)
        self.edges = [
            np.linspace(-1, 1, self.bins[0] - 1), np.linspace(-1, 1, self.bins[1] - 1),
            np.linspace(-np.pi, np.pi, self.bins[2] - 1), np.linspace(-1, 1, self.bins[3] - 1),
        ]

    def state(self, obs):
        compact = (obs[0], obs[1], np.arctan2(obs[3], obs[2]), obs[4])
        return tuple(int(np.digitize(value, edge)) for value, edge in zip(compact, self.edges))

    def act(self, state, greedy=False):
        if not greedy and self.rng.random() < self.epsilon:
            return int(self.rng.integers(self.actions))
        values = self.q[state]
        return int(self.rng.choice(np.flatnonzero(values == values.max())))

    def update(self, state, action, reward, next_state, terminal):
        target = reward if terminal else reward + GAMMA * self.q[next_state].max()
        td_error = target - self.q[state + (action,)]
        self.q[state + (action,)] += ALPHA * td_error
        return float(td_error ** 2)

    def decay(self, episode):
        fraction = min(1.0, episode / max(1, EPSILON_DECAY_EPISODES))
        self.epsilon = EPSILON_START + fraction * (EPSILON_END - EPSILON_START)


## 5. Checkpoint, metrics, and automatic resume


In [ ]:
import csv, os, pickle, time
from datetime import datetime, timezone

FIELDS = ["timestamp", "episode", "total_steps", "reward", "loss", "epsilon", "episode_length", "steps_per_second", "success"]

def save_state(agent, episode, total_steps):
    payload = {"version": 1, "episode": episode, "total_steps": total_steps,
               "q": agent.q, "epsilon": agent.epsilon, "rng": agent.rng.bit_generator.state,
               "bins": agent.bins}
    path = CHECKPOINT_DIR / f"checkpoint_{total_steps:012d}.pkl"
    temp = path.with_suffix(".tmp")
    with temp.open("wb") as file:
        pickle.dump(payload, file, pickle.HIGHEST_PROTOCOL); file.flush(); os.fsync(file.fileno())
    temp.replace(path)
    for old in sorted(CHECKPOINT_DIR.glob("checkpoint_*.pkl"))[:-3]: old.unlink()
    return path

def restore_state(agent):
    for path in sorted(CHECKPOINT_DIR.glob("checkpoint_*.pkl"), reverse=True):
        try:
            with path.open("rb") as file: state = pickle.load(file)
            if state["version"] != 1 or tuple(state["bins"]) != agent.bins: raise ValueError("incompatible checkpoint")
            agent.q[:] = state["q"]; agent.epsilon = state["epsilon"]
            agent.rng.bit_generator.state = state["rng"]
            print("Resumed", path.name); return state["episode"], state["total_steps"]
        except Exception as error: print("Skipped", path.name, error)
    return 0, 0

def log_row(row):
    new = not METRICS_PATH.exists()
    with METRICS_PATH.open("a", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=FIELDS)
        if new: writer.writeheader()
        writer.writerow(row); file.flush(); os.fsync(file.fileno())


## 6. Train
Set `SMOKE_TEST=True` first. Resume is automatic when configuration is compatible.


In [ ]:
env = SinglePendulumEnv(True, MAX_EPISODE_STEPS, ACTION_LIMIT)
agent = QLearningAgent(BINS, env.action_space.n, SEED)
start_episode, total_steps = restore_state(agent)
clock, start_steps = time.perf_counter(), total_steps
try:
    for episode in range(start_episode + 1, EPISODES + 1):
        obs, _ = env.reset(seed=SEED + episode); state = agent.state(obs)
        reward_sum, losses, success = 0.0, [], False
        for length in range(1, MAX_EPISODE_STEPS + 1):
            action = agent.act(state)
            next_obs, reward, terminated, truncated, info = env.step(action)
            next_state = agent.state(next_obs)
            losses.append(agent.update(state, action, reward, next_state, terminated))
            state = next_state; reward_sum += reward; total_steps += 1
            success |= abs(info["theta"]) < 0.35
            if terminated or truncated: break
        agent.decay(episode)
        elapsed = max(time.perf_counter() - clock, 1e-9)
        log_row({"timestamp": datetime.now(timezone.utc).isoformat(), "episode": episode,
                 "total_steps": total_steps, "reward": reward_sum, "loss": np.mean(losses),
                 "epsilon": agent.epsilon, "episode_length": length,
                 "steps_per_second": (total_steps-start_steps)/elapsed, "success": int(success)})
        if episode % CHECKPOINT_EVERY == 0: print("Saved", save_state(agent, episode, total_steps))
        if episode % 10 == 0: print(episode, round(reward_sum, 1), round(agent.epsilon, 3))
finally:
    save_state(agent, episode if "episode" in locals() else start_episode, total_steps)


## 7. Static dashboard


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

metrics = pd.read_csv(METRICS_PATH).drop_duplicates("episode", keep="last").sort_values("episode")
window = min(20, len(metrics))
fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
fig.suptitle(TITLE, fontsize=16, fontweight="bold")
axes[0, 0].plot(metrics.episode, metrics.reward, alpha=.3)
axes[0, 0].plot(metrics.episode, metrics.reward.rolling(window, min_periods=1).mean())
axes[0, 0].set_title("Reward and moving average")
axes[0, 1].plot(metrics.episode, metrics.loss); axes[0, 1].set_title("Training loss")
axes[0, 2].plot(metrics.episode, metrics.epsilon); axes[0, 2].set_title("Epsilon")
axes[1, 0].plot(metrics.episode, metrics.episode_length); axes[1, 0].set_title("Episode length")
axes[1, 1].plot(metrics.episode, metrics.steps_per_second); axes[1, 1].set_title("Steps/second")
axes[1, 2].plot(metrics.episode, metrics.success); axes[1, 2].set_title("Upright reached")
for axis in axes.flat:
    axis.set_xlabel("Episode"); axis.grid(alpha=.25)
fig.savefig(DASHBOARD_PATH, dpi=160)
plt.show()
print("Saved", DASHBOARD_PATH)


## 8. Greedy evaluation
This does not update the table or epsilon.


In [ ]:
evaluation_rewards = []
for episode in range(10):
    obs, _ = env.reset(seed=10_000 + episode); state = agent.state(obs); total = 0.0
    while True:
        obs, reward, terminated, truncated, info = env.step(agent.act(state, greedy=True))
        state = agent.state(obs); total += reward
        if terminated or truncated: break
    evaluation_rewards.append(total)
print("Greedy reward mean/std:", np.mean(evaluation_rewards), np.std(evaluation_rewards))


## Replay The Trained Run

Colab cannot reliably open MuJoCo's interactive desktop viewer. This block runs a
deterministic evaluation, streams rendered frames directly into an MP4, saves it in
the run directory, and displays it inline. It replays the current trained policy or
controller; it is not an exact recording of a stochastic training episode.


In [ ]:
import imageio.v2 as imageio
from IPython.display import Video, display

REPLAY_SEED, REPLAY_SECONDS, REPLAY_FPS = 50_001, 10.0, 50
REPLAY_PATH = OUTPUT_DIR / "replay.mp4"
replay_env = SinglePendulumEnv(True, int(REPLAY_SECONDS / 0.005), ACTION_LIMIT)
observation, _ = replay_env.reset(seed=REPLAY_SEED)
renderer = mujoco.Renderer(replay_env.model, height=480, width=640)
writer = imageio.get_writer(REPLAY_PATH, fps=REPLAY_FPS, codec="libx264", quality=8)
frame_stride = max(1, round(1 / (replay_env.dt * REPLAY_FPS)))
try:
    for step in range(replay_env.max_episode_steps):
        action = agent.act(agent.state(observation), greedy=True)
        observation, _, terminated, truncated, _ = replay_env.step(action)
        if step % frame_stride == 0:
            renderer.update_scene(replay_env.data, camera="replay")
            writer.append_data(renderer.render())
        if terminated or truncated:
            break
finally:
    writer.close()
    renderer.close()
print("Saved replay:", REPLAY_PATH)
display(Video(str(REPLAY_PATH), embed=True))
